# 1. Data Cleaning and Preprocessing

In this notebook, we will perform the initial data cleaning and preprocessing steps for the HUPA-UC Diabetes Dataset. This includes loading the data, merging the different files, handling missing values and inconsistencies, and preparing the data for exploratory data analysis.

In [1]:
import pandas as pd
import numpy as np
import os
import glob

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


## 1.1 Load Data

In [2]:
# Load the demographic data
demographics_df = pd.read_csv('data/HUPA-UC Diabetes Dataset/T1DM_patient_sleep_demographics_with_race.csv')

In [3]:
# Load the time-series data
time_series_files = glob.glob('data/HUPA-UC Diabetes Dataset/HUPA*.csv')

all_dfs = []
for file in time_series_files:
    df = pd.read_csv(file, delimiter=';')
    patient_id = os.path.basename(file).split('.')[0]
    df['Patient_ID'] = patient_id
    all_dfs.append(df)

time_series_df = pd.concat(all_dfs, ignore_index=True)

## 1.2 Merge Data

In [4]:
merged_df = pd.merge(time_series_df, demographics_df, on='Patient_ID')

In [5]:
merged_df.to_csv('master_df.csv', index=False)

## 1.3 Handle Data Types

In [6]:
master_df = pd.read_csv('master_df.csv')
master_df['time'] = pd.to_datetime(master_df['time'])
numeric_cols = ['glucose', 'calories', 'heart_rate', 'steps', 'basal_rate', 'bolus_volume_delivered', 'carb_input', 'Age', 'Average Sleep Duration (hrs)', 'Sleep Quality (1-10)', '% with Sleep Disturbances']
for col in numeric_cols:
    master_df[col] = pd.to_numeric(master_df[col], errors='coerce')

## 1.4 Handle Missing Values

In [7]:
missing_values = master_df.isnull().sum()
print(missing_values[missing_values > 0])

Series([], dtype: int64)


Now, we need to decide on a strategy for handling these missing values. We will use forward fill for the time-series data and mean/median imputation for the demographic data.

In [8]:
time_series_cols = ['glucose', 'calories', 'heart_rate', 'steps', 'basal_rate', 'bolus_volume_delivered', 'carb_input']
for col in time_series_cols:
    master_df[col] = master_df[col].fillna(method='ffill')

demographic_cols = ['Age', 'Average Sleep Duration (hrs)', 'Sleep Quality (1-10)', '% with Sleep Disturbances']
for col in demographic_cols:
    master_df[col] = master_df[col].fillna(master_df[col].median())

/var/folders/0c/trjhsp1d0ld2klcmhwxtch2w0000gn/T/ipykernel_17117/4033907751.py:3: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  master_df[col] = master_df[col].fillna(method='ffill')


## 1.5 Save Cleaned Data

In [9]:
master_df.to_csv('data/cleaned_diabetes_data.csv', index=False)